In [126]:
import pandas as pd
from yaml import safe_dump, safe_load

from rfm_sp500 import constants, validation

with open(constants.CONFIG_FILE, "r") as f:
    config = safe_load(f)

start_date = config["start_date"]
end_date_exclusive = config["end_date_exclusive"]

In [127]:
sp_dir = constants.RAW_DATA_DIR / f"sp500_data_{start_date}_{end_date_exclusive}.csv.gz"
sp500_df = pd.read_csv(sp_dir, parse_dates=["Date"])
empty_tickers = ["FDXF", "HONA", "Q", "SNDK", "VYLR"]
sp500_df = sp500_df[~sp500_df["Ticker"].isin(empty_tickers)]
sp500_df.head()

,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,A,114.009102,118.639999,0.194,120.089996,117.110001,118.940002,0.0,2030700.0
1,2021-01-05,A,114.941277,119.610001,0.000,120.320000,118.070000,118.480003,0.0,2344000.0
2,2021-01-06,A,118.093231,122.889999,0.000,123.879997,118.209999,118.690002,0.0,2428500.0
3,2021-01-07,A,121.235580,126.160004,0.000,127.480003,124.110001,124.330002,0.0,1775900.0
4,2021-01-08,A,122.100464,127.059998,0.000,127.639999,125.849998,126.489998,0.0,1577200.0


In [128]:
# Check for duplicates

duplicate_ticker_date = sp500_df[sp500_df.duplicated(subset=["Ticker", "Date"], keep=False)]
duplicate_ticker_date

,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume


No duplicates

In [129]:
def flatten(x):
    summary, middle = validation.get_nan_summary(x)
    return pd.Series({**summary, **{f"middle_{col}": n for col, n in middle.items()}})


nan_summary_df = sp500_df.groupby("Ticker").apply(flatten)
nan_summary_df = nan_summary_df.loc[~nan_summary_df.eq(0).all(axis=1)].reset_index()

nan_save_dir = constants.PROCESSED_DATA_DIR / "nan_summary_sp500.csv.gz"

nan_summary_df.to_csv(nan_save_dir, index=False)
print(f"Saved S&P 500 NaN summary to {nan_save_dir}")

nan_summary_df = nan_summary_df.replace(0, pd.NA)
nan_summary_df = nan_summary_df.dropna(axis=1)
nan_summary_df

Saved S&P 500 NaN summary to D:\dev\crm-rfm-sp500\data\processed\nan_summary_sp500.csv.gz


,Ticker,Leading
0,APP,70
1,CEG,263
2,COIN,69
3,EXE,26
4,GEHC,492
5,GEV,812
6,HOOD,143
7,KVUE,587
8,RDDT,808
9,SOLV,811


There is only data missing leading data meaning these 11 stocks were added after the start date and there is no trailing missing data or data missing in the middle

Therefore, can focus on deciding how to handle unequal history from leading values only

In [130]:
leading_tickers = nan_summary_df["Ticker"].unique()

In [131]:
benchmark_dir = constants.RAW_DATA_DIR / f"benchmark_data_{start_date}_{end_date_exclusive}.csv.gz"
benchmark_df = pd.read_csv(benchmark_dir, parse_dates=["Date"])
benchmark_df.head()

,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,3700.649902,3700.649902,0.0,3769.989990,3662.709961,3764.610107,0.0,5015000000
1,2021-01-05,3726.860107,3726.860107,0.0,3737.830078,3695.070068,3698.020020,0.0,4591020000
2,2021-01-06,3748.139893,3748.139893,0.0,3783.040039,3705.340088,3712.199951,0.0,6064110000
3,2021-01-07,3803.790039,3803.790039,0.0,3811.550049,3764.709961,3764.709961,0.0,5099160000
4,2021-01-08,3824.679932,3824.679932,0.0,3826.689941,3783.600098,3815.050049,0.0,4773040000


In [132]:
benchmark_duplicate_ticker_date = benchmark_df[benchmark_df.duplicated(subset=["Date"], keep=False)]
benchmark_duplicate_ticker_date

,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume


In [133]:
nan_save_dir = constants.PROCESSED_DATA_DIR / "nan_summary_benchmark.yaml"

with open(nan_save_dir, "w") as f:
    safe_dump(validation.get_nan_summary(benchmark_df), f, sort_keys=False)
    print(f"Saved benchmark NaN summary to {nan_save_dir}")

Saved benchmark NaN summary to D:\dev\crm-rfm-sp500\data\processed\nan_summary_benchmark.yaml
